# module-extra-repr — ex2: nested module repr — parent extra_repr above an indented child

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-extra-repr`. Running the final beacon cell reports progress against the `PyTorch: Module __repr__` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Module __repr__` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-extra-repr`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-extra-repr"
DD_SUBTOPIC = "PyTorch: Module __repr__"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `extra_repr` — nested module edition

When a Module has child modules registered as attributes, `print(parent)` auto-indents and prints each child's `__repr__` on its own line — this is PyTorch's tree-pretty-printer:

```
MyMLP(
  hidden_dim=4
  (linear): MyReprLinear(in_features=3, out_features=4, bias=True)
  (act): ReLU()
)
```

The previous drill (ex1) implemented `extra_repr` on a **flat** Linear-style module. This drill exercises the same operator on a **nested** parent that owns a child Linear — confirming the parent's `extra_repr` appears at the TOP and the child's full repr appears INDENTED under it.

### Exercise 2 — nested module repr — parent extra_repr above an indented child

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `extra_repr` to a parent Module that owns a registered child Linear, and verify the child's full repr appears INDENTED beneath the parent's extra_repr in the default tree print.
> Keywords: extra_repr, nested-module, child-indentation, tree-print
> ```

**KCs targeted:** `extra-repr-returns-string`, `extra-repr-shows-up-in-nested-print`

Implement `ex2_build_mlp(in_features, hidden_dim, out_features)`. Build a two-layer MLP-shaped parent module that stores `hidden_dim` as the ONLY thing surfaced via `extra_repr`, and owns a registered child `nn.Linear`:

```python
class MyMLP(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_features)
    def extra_repr(self):
        return f'hidden_dim={self.hidden_dim}'
    def forward(self, x):
        return self.fc2(t.relu(self.fc1(x)))
```

Then return an instance from `ex2_build_mlp(...)`.

**Why this is the deepening facet.** The test verifies BOTH that `extra_repr` contains `hidden_dim=...` AND that `repr(model)` contains the child Linear's `(fc1): Linear(...)` line with INDENTATION (the leading two-space indent that PyTorch's `__repr__` inserts under a parent). The child contribution comes for free from `nn.Module.__repr__` recursion — the drill confirms `extra_repr` doesn't break that recursion.

In [ ]:
def ex2_build_mlp(in_features: int, hidden_dim: int, out_features: int):
    """Return a MyMLP instance: extra_repr surfaces hidden_dim; owns fc1, fc2."""
    raise NotImplementedError()


def _test_ex2():
    model = ex2_build_mlp(in_features=3, hidden_dim=8, out_features=2)
    import torch.nn as nn

    assert isinstance(model, nn.Module), 'must be an nn.Module'

    # extra_repr surfaces hidden_dim only.
    extra = model.extra_repr()
    assert isinstance(extra, str), f'extra_repr must return str, got {type(extra)}'
    assert 'hidden_dim=8' in extra, f'extra_repr must contain hidden_dim=8, got {extra!r}'

    # Repr contains both the parent extra_repr AND each child registered as a sub-module.
    r = repr(model)
    assert 'hidden_dim=8' in r, f'parent extra_repr must appear in repr, got:\n{r}'
    assert '(fc1):' in r, f'child fc1 must appear under parent, got:\n{r}'
    assert '(fc2):' in r, f'child fc2 must appear under parent, got:\n{r}'
    assert 'Linear' in r, 'child Linear class name must appear'

    # Critical: child repr must be INDENTED (two-space leading per PyTorch's tree printer).
    lines = r.split('\n')
    fc1_line = next(L for L in lines if '(fc1):' in L)
    assert fc1_line.startswith('  '), (
        f'child line must be indented (PyTorch nested tree print), got: {fc1_line!r}'
    )

    # Forward must still work end-to-end (the module is a real MLP, not just a repr container).
    x = t.randn(4, 3)
    y = model(x)
    assert y.shape == (4, 2), f'forward shape: expected (4,2), got {tuple(y.shape)}'

    # Parameters must be registered (the child Linears should each contribute 2 tensors).
    params = list(model.parameters())
    assert len(params) == 4, f'expected 4 params (W1, b1, W2, b2), got {len(params)}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_build_mlp(in_features: int, hidden_dim: int, out_features: int):
    import torch.nn as nn
    class MyMLP(nn.Module):
        def __init__(self, in_features, hidden_dim, out_features):
            super().__init__()
            self.hidden_dim = hidden_dim
            self.fc1 = nn.Linear(in_features, hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, out_features)
        def extra_repr(self):
            return f'hidden_dim={self.hidden_dim}'
        def forward(self, x):
            return self.fc2(t.relu(self.fc1(x)))
    return MyMLP(in_features, hidden_dim, out_features)
```

**The recursion is free — but `extra_repr` MUST return a string.** If you accidentally return a tensor or `None`, the default `nn.Module.__repr__` will raise during printing. Always: `return f'k=v, ...'`.

**Indentation comes from `nn.Module.__repr__`.** It calls `_addindent` on each child's repr and prefixes `(name): ` — you do NOT need to format the child yourself. Your `extra_repr` lives only on its own line; the children fall through to PyTorch's tree printer.

**Difference from ex1.** ex1 verified the FLAT case (`bias=True/False` as a boolean, not a tensor). ex2 verifies the NESTED case — that `extra_repr` plays nicely with registered child modules in the recursive print.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()